In [16]:
import copy
import time
import pickle

In [17]:
%run helpers.ipynb

In [18]:
m=3
heis_dim=2*m+1

# Bases

In [19]:
# # If the file 'T_symb_basis' has already been written correctly,
# # there's no need to run the next cell, just pickle.load

# file=open('T_symb_basis_m%d'%m,'rb') ## 'rb' means 'read binary mode'
# T_symb_basis=pickle.load(file)
# file.close()

# Y=T_symb_basis[0]
# H=T_symb_basis[1]
# E=T_symb_basis[2]
# X=T_symb_basis[3]
# for i in range(1,2*m+1):
#     globals()['e%d'%i]=T_symb_basis[i+3]
# N=T_symb_basis[len(T_symb_basis)-1]

In [20]:
# Constructing bases for pickling; no need if already pickled

Y=T_symb_basis_elt('Y',1,2*m+1)
H=T_symb_basis_elt('H',0,2*m+1)
E=T_symb_basis_elt('E',0,2*m+1)
X=T_symb_basis_elt('X',-1,2*m+1)
for i in range(1,2*m+1):
    globals()['e%d'%i]=T_symb_basis_elt('e%d'%i,-i,2*m+1)
N=T_symb_basis_elt('N',-7,2*m+1)

In [21]:
heis_basis=[globals()['e%d'%i] for i in range(1,2*m+1)]+[N]
V_basis=heis_basis[0:len(heis_basis)-1]
gl2_basis=[Y,H,E,X]
T_symb_basis=gl2_basis+heis_basis
T_symb_basis_strs=[str(A) for A in T_symb_basis]
T_symb_basis_dict={str(A): A for A in T_symb_basis}

# ad action

In [26]:
#  No need to run this if T_symb_basis has been pickled
#  Set ext_ad_dicts
#  The dual representation of ext(T_symb)

# First, reset the dicts
for A in T_symb_basis:
    for i in range(len(A.ext_ad_dicts)):
        A.ext_ad_dicts[i]={}

# Set degree 0 and 1 dicts
for A in T_symb_basis:
    A.ext_ad_dicts[0]={}
    for B in T_symb_basis:
        temp=ad(A,B)
        if temp!=0:
            for i in range(len(T_symb_basis)):
                if (T_symb_basis_strs[i],) in A.ext_ad_dicts[1]:
                    A.ext_ad_dicts[1][(T_symb_basis_strs[i],)]+=ext_T_symb_elt({(str(B),):-temp.vec_rep[i]},heis_dim)
                else:
                    A.ext_ad_dicts[1][(T_symb_basis_strs[i],)]=ext_T_symb_elt({(str(B),):-temp.vec_rep[i]},heis_dim)

# Set higher degree dicts
for deg in [2,3]:
    for A in T_symb_basis:
        for B in ext_basis[deg]:
            # For each element (B1,B2) in B, where B2 has degree 1,
            # compute ad(A)(B1) wedge B2 + B1 wedge ad(A)(B2),
            # where ad(A) the dual/wedge representation 
            B1=B[0:len(B)-1]
            ext_B1=ext_T_symb_elt({B1:1},heis_dim)
            B2=B[len(B)-1:len(B)]
            ext_B2=ext_T_symb_elt({B2:1},heis_dim)
            A.ext_ad_dicts[deg][B]=(A.ext_ad_dicts[deg-1][B1]).wedge(ext_B2) + ext_B1.wedge(A.ext_ad_dicts[1][B2])

In [27]:
#  No need to run this if T_symb_basis has been pickled
# Set cochain_ad_dicts
#  Before running this, set ext_ad_dicts (the cell above)

time0=time.time()

# C0 is just T_symb
for A in T_symb_basis:
    for B in A.ad_dict:
        A.cochain_ad_dicts[0][(B,)]=convert_T_symb_elt_to_cochain(ad(A,globals()[B]))

# higher deg cochains
for deg in range(1,4):
    for A in T_symb_basis: #T_symb_basis_elt
        for B1 in ext_basis[deg]: # tuple of strings
            for B2 in T_symb_basis: # T_symb_basis_elt
                key=B1+(str(B2),)
                cochain_B2=convert_T_symb_elt_to_cochain(B2)
                cochain_ad_B2=convert_T_symb_elt_to_cochain(A.ad_dict[str(B2)])
                t1=ext_T_symb_elt({B1:1},heis_dim).wedge(cochain_ad_B2)
                t2=A.ext_ad_dicts[deg][B1].wedge(cochain_B2)
                A.cochain_ad_dicts[deg][key]=t1+t2             
time1=time.time()
print('cochain_ad_dicts set\ntotal time: '+hrs_min_sec(time1-time0))

cochain_ad_dicts set
total time: 3.0 sec


In [28]:
def ext_ad(se,ext):
    '''se: a T_symb_elt object
       ext: an ext_T_symb_elt object
       returns: an ext_T_symb_elt object representing ad(se,ext)'''
    
    if ext==0 or se==0: return ext_T_symb_elt({},heis_dim)
    if ext.coeff_dict=={} or se==T_symb_elt({},heis_dim): return ext_T_symb_elt({},heis_dim)
    
    result=ext_T_symb_elt({},heis_dim)
    for ext_key in ext.coeff_dict:
        deg=ext_T_symb_elt_deg(ext_T_symb_elt({ext_key:1},heis_dim))
        for i in range(len(T_symb_basis)):
            if se.vec_rep[i]!=0:
                result+=se.vec_rep[i]*ext.coeff_dict[ext_key]*T_symb_basis[i].ext_ad_dicts[deg][ext_key]
    return result

In [29]:
def cochain_ad(se,c):
    '''se: a T_symb_elt object
       c: a homogeneous cochain object
       returns: a cochain object representing ad(se,c)'''
    
    if c==0 or se==0: return cochain({},heis_dim)
    if c.coeff_dict=={} or se==T_symb_elt({},heis_dim): return cochain({},heis_dim)
    
    result=cochain({},heis_dim)
    for c_key in c.coeff_dict:
        deg=cochain_deg(cochain({c_key:1},heis_dim))
        for i in range(len(T_symb_basis)):
            if se.vec_rep[i]!=0:
                result+=se.vec_rep[i]*c.coeff_dict[c_key]*T_symb_basis[i].cochain_ad_dicts[deg][c_key]
    return result

In [30]:
# # pickle T_symb_basis with its ad_dict, ext_ad_dicts, cochain_ad_dicts attributes

# file=open('T_symb_basis_m%d'%m,'wb') # 'wb' means 'write binary mode'
# pickle.dump(T_symb_basis,file)
# file.close()

# Inner Products

We consider the inner product on the Tanaka symbol with orthonormal basis $(Y,H,E,X,\varepsilon_1,\ldots,\varepsilon_{2m},\eta)$ and lengths

$$|Y|^2=|X|^2=1,|H|^2=|E|^2=2,|\varepsilon_i|^2=\frac{(i-1)!}{(2m-i)!}, |\eta|^2=1$$

along with the innerproduct induced on tensor spaces. In particular, 

$$|A^*\wedge B^*\otimes C|^2 = \frac{|C|^2}{|A|^2|B|^2}$$

In [33]:
# Set the list used in T_symb_iprod
from sympy import factorial
T_symb_iprod_list=[1,2,2,1]+[factorial(i-1)/factorial(2*m-i) for i in range(1,2*m+1)]+[1]

In [35]:
# Set the lists used in ext_iprod
from sympy import Rational
ext_iprod_lists=[None,[Rational(1,A) for A in T_symb_iprod_list],[0]*len(ext_basis[2]),[0]*len(ext_basis[3])]
for ext1 in ext_basis[2]:
    ind1=T_symb_basis_strs.index(ext1[0])
    ind2=T_symb_basis_strs.index(ext1[1])
    ext_iprod_lists[2][ext_basis[2].index(ext1)]=ext_iprod_lists[1][ind1]*ext_iprod_lists[1][ind2]

for ext1 in ext_basis[3]:
    ind1=T_symb_basis_strs.index(ext1[0])
    ind2=T_symb_basis_strs.index(ext1[1])
    ind3=T_symb_basis_strs.index(ext1[2])
    result=ext_iprod_lists[1][ind1]*ext_iprod_lists[1][ind2]*ext_iprod_lists[1][ind3]
    ext_iprod_lists[3][ext_basis[3].index(ext1)]=result

In [36]:
# Set the lists used in cochain_iprod
cochain_iprod_lists=[T_symb_iprod_list] + [[0]*len(cochain_basis[i]) for i in range(1,4)]
for deg in range(1,4):
    for c1 in cochain_basis_tuples[deg]:
        ext_len_sq=ext_iprod_lists[deg][ext_basis[deg].index(c1[0:len(c1)-1])]
        T_elt_len_sq=T_symb_iprod_list[T_symb_basis_strs.index(c1[(len(c1))-1])]
        cochain_iprod_lists[deg][cochain_basis_tuples[deg].index(c1)]=ext_len_sq*T_elt_len_sq

In [37]:
def T_symb_iprod(t1,t2):
    '''t1,t2: T_symb_elt objects
       returns: the inner product of t1 and t2'''
    return(sum([t1.vec_rep[i]*t2.vec_rep[i]*T_symb_iprod_list[i] 
                for i in range(len(T_symb_basis))]))

In [38]:
def ext_iprod(ext1,ext2):
    '''ext1,ext2: ext_T_symb_elt objects
       returns: the inner product of ext1 and ext2, or None if the elements have differing degrees'''
    result=0
    for A in ext1.coeff_dict:
        deg=ext_T_symb_elt_deg(ext_T_symb_elt({A:1},heis_dim))
        if A in ext2.coeff_dict:
            A_index=ext_basis[deg].index(A)
            result+=ext1.coeff_dict[A]*ext2.coeff_dict[A]*ext_iprod_lists[deg][A_index]
    return result

In [39]:
def cochain_iprod(c1,c2):
    '''c1,c2: cochain objects
       returns: the inner product of c1 and c2'''
    result=0
    for A in c1.coeff_dict:
        deg=cochain_deg(cochain({A:1},heis_dim))
        if A in c2.coeff_dict:
            A_index=cochain_basis_tuples[deg].index(A)
            result+=c1.coeff_dict[A]*c2.coeff_dict[A]*cochain_iprod_lists[deg][A_index]
    return result

# Coboundary Operator

Recall that the coboundary operator is

$$\partial: C^k(\mathfrak{m},\mathfrak{g})\to C^{k+1}(\mathfrak{m},\mathfrak{g})$$

and is defined by

$$\partial\phi(\alpha_0,\ldots,\alpha_{k}) = \sum_{i=0}^{k}(-1)^i\big[\alpha_i,\phi(\alpha_0,\ldots,\hat\alpha_i,\ldots, \alpha_{k})\big]$$
$$+ \sum_{i<j}(-1)^{i+j}\phi([\alpha_i,\alpha_j],\alpha_0,\ldots, \hat \alpha_i,\ldots,\hat\alpha_j,\ldots,\alpha_{k})$$


In [134]:
# Unpickle coboundary_dict

file=open('coboundary_dict_m%d'%m,'rb') ## 'rb' means 'read binary mode'
coboundary_dict=pickle.load(file)
file.close()

In [135]:
# No need to run this if coboundary_dict has been pickled

coboundary_dict={}
# keys: tuples representing elementary cochains of deg 0,1,2
# values: cochain objects representing coboundary(key)

time0=time.time()
for deg in [0,1,2]:
    for c in cochain_basis[deg]:
        c_tuple=cochain_basis_tuples[deg][cochain_basis[deg].index(c)]
        dc=cochain({},heis_dim)
        for ext_elt in ext_basis[deg+1]:
            dc_ext_elt=T_symb_elt([0]*len(T_symb_basis),heis_dim)

            # First, compute im:=dc(ext_elt), a degree zero cochain
            for i in range(len(ext_elt)):
                # (-1)**i*[ai,c(a0,...\hat ai,...ak)]
                no_i=ext_T_symb_elt({tuple(ext_elt[0:i]
                                           +ext_elt[i+1:len(ext_elt)]):(-1)**i},heis_dim)
                im=apply_cochain_map(c,no_i)
                dc_ext_elt+=ad(globals()[ext_elt[i]],im)
            for i in range(len(ext_elt)):
                for j in range(i+1,len(ext_elt)):
                    # (-1)**(i+j)*c([ai,aj],a0,...,\hat ai,...\hat aj,...ak)
                    ext_elt2=ext_T_symb_elt({ext_elt[0:i]+ext_elt[i+1:j]
                                             +ext_elt[j+1:len(ext_elt)]:1},heis_dim)
                    T_elt1=ad(T_symb_basis_dict[ext_elt[i]],T_symb_basis_dict[ext_elt[j]])
                    ext_elt1=ext_T_symb_elt({(T_symb_basis_strs[i],):T_elt1.vec_rep[i] 
                                             for i in range(len(T_symb_basis))},heis_dim)
                    # wedging with the zero cochain gives zero, so treat deg=1 separately
                    if ext_elt2==0:
                        new_wedge=ext_elt1
                    else:
                        new_wedge=ext_elt1.wedge(ext_elt2)
                    dc_ext_elt+=apply_cochain_map((-1)**(i+j)*c,new_wedge)
            dc+=ext_T_symb_elt({ext_elt:1},heis_dim).wedge(dc_ext_elt)
        coboundary_dict[c_tuple]=dc
time1=time.time()        
print('coboundary_dict set\ntotal time: '+hrs_min_sec(time1-time0))

coboundary_dict set
total time: 45.0 sec


In [44]:
# # pickle coboundary_dict

# file=open('coboundary_dict_m%d'%m,'wb') # 'wb' means 'write binary mode'
# pickle.dump(coboundary_dict,file)
# file.close()

In [140]:
# No need to run this if coboundary_image_basis has been pickled
# Sort first by degree, then by weight

time0=time.time()
coboundary_image_basis=[{},{},{},{}]
for A in coboundary_dict.values():
    A.wght=cochain_wght(A)
    A.deg=cochain_deg(A)
    if A.deg!='Nil':
        if A.wght in coboundary_image_basis[A.deg]:
            coboundary_image_basis[A.deg][A.wght].append(A)
        else: coboundary_image_basis[A.deg][A.wght]=[A]
            
# perform row reduction
for deg in range(len(coboundary_image_basis)):
    for wght in coboundary_image_basis[deg]:
        coboundary_image_basis[deg][wght]=find_cochain_basis(coboundary_image_basis[deg][wght])
        
time1=time.time()
print('coboundary_image_basis set\ntotal time:',hrs_min_sec(time1-time0))

coboundary_image_basis set
total time: 11.0 sec


In [46]:
# # pickle coboundary_image_basis

# file=open('coboundary_image_basis_m%d'%m,'wb') # 'wb' means 'write binary mode'
# pickle.dump(coboundary_image_basis,file)
# file.close()

In [47]:
# unpickle coboundary_image_basis

file=open('coboundary_image_basis_m%d'%m,'rb') ## 'rb' means 'read binary mode'
coboundary_image_basis=pickle.load(file)
file.close()

In [48]:
def ortho_to_coboundary_image(c_list):
    '''c_list: a list of cochains
       result: True if each element of c_list is orthogonal 
       to the image of the coboundary, False otherwise'''
    
    for c in c_list:
        c.deg=cochain_deg(c)
        c.wght=cochain_wght(c)
        if c.deg=='UNKNOWN' or c.wght=='UNKNOWN':
            for A in coboundary_image_basis:
                if cochain_iprod(A,c)!=0:
                    return False
        if c.wght!='Nil':
            if c.wght in coboundary_image_basis[c.deg]:
                for A in coboundary_image_basis[c.deg][c.wght]:
                    if cochain_iprod(A,c)!=0:
                        return False
        return True

# Normalization Condition

In [49]:
## This may all be obselete now that I'm rethinking the overall algorithm

norm_cond_basis=[]

$\wedge^2V^*\otimes V$-portion of normalization condition

In [50]:
def convert_e_exp_to_cochain(A):
    if type(A)==type(e[1,1,1]):
        T_elt=(T_symb_basis[3+A.indices[0]],T_symb_basis[3+A.indices[1]],T_symb_basis[3+A.indices[2]])
        return cochain({T_elt:1})
    result=cochain({})
    for B in dict(A.as_coefficients_dict()):
        T_elt=(T_symb_basis[3+B.indices[0]],T_symb_basis[3+B.indices[1]],T_symb_basis[3+B.indices[2]])
        result+=cochain({T_elt:A.as_coefficients_dict()[B]})
    return result

In [51]:
e=IndexedBase('e',shape=(2*m,2*m,2*m))
file=open('eijk_norm_cond_m%d'%m,'rb') ## 'rb' means 'read binary mode'
temp=pickle.load(file)
file.close()

VVV_norm_cond=[convert_e_exp_to_cochain(A) for A in temp]
norm_cond_basis+=VVV_norm_cond

TypeError: unhashable type: 'T_symb_basis_elt'

In [ ]:
ortho_to_coboundary_image(VVV_norm_cond)

$\wedge^2V^*\otimes \mathfrak{a}$-portion of normalization condition

In [ ]:
VVa_norm_cond=[]

for i in range(len(V_basis)):
    ei=V_basis[i]
    for j in range(i+1,len(V_basis)):
        ej=V_basis[j]
        if j!=2*m-1-i:
            for gl2_elt in gl2_basis:
                VVa_norm_cond.append(cochain({(ei,ej,gl2_elt):1}))
for i in range(1,m):
    ei=V_basis[i-1]
    ej=V_basis[2*m-i]
    for gl2_elt in gl2_basis:
        VVa_norm_cond.append(cochain({(V_basis[i-1],V_basis[2*m-i],gl2_elt):1,
                                  (V_basis[i],V_basis[2*m-i-1],gl2_elt):1}))

norm_cond_basis+=VVa_norm_cond

In [ ]:
ortho_to_coboundary_image(VVa_norm_cond)

$\mathbb{R}X^*\wedge V^*\otimes V$-portion of normalization condition

In [ ]:
## I guess this is wrong. To do: Fix!

XVV_norm_cond=[sum([Rational(factorial(i+k-1)*factorial(2*m-i)*factorial(2*m-k-1),
                            factorial(i-1)*factorial(k)*factorial(2*m-i-k)*factorial(2*m-1))
                   *cochain({(V_basis[i+k-1],V_basis[k]):1}) for k in range(0,2*m-i+1)]) 
               for i in range(1,2*m+1)]

In [ ]:
cochain_ad(Y,XVV_norm_cond[0])

In [ ]:
ortho_to_coboundary_image([XVV_norm_cond[0]])

$\mathbb{R}X^*\wedge V^*\otimes \mathfrak{a}$-portion of normalization condition

In [ ]:
XVa_norm_cond=[cochain({(X,V_basis[2*m-1],E):1}),
              cochain({(X,V_basis[2*m-1],Y):1}),
              cochain({(X,V_basis[2*m-1],H):2*m-1,(X,V_basis[2*m-2],Y):2})]

norm_cond_basis+=XVa_norm_cond

In [ ]:
ortho_to_coboundary_image(XVa_norm_cond)

$\mathbb{R}\eta^*\wedge V^*\otimes V$-portion of normalization condition

In [ ]:
a_perp=[]
for i in range(len(V_basis)):
    for j in range(len(V_basis)):
        if j not in [i-1,i,i+1]:
            a_perp.append(cochain({(V_basis[i],V_basis[j]):1}))

for i in range(2,len(V_basis)):
    a_perp.append(cochain({(V_basis[i-1],V_basis[i]):1,
                           (V_basis[0],V_basis[1]):-Rational(i*(2*m-i),2*m-1)}))

for i in range(3,2*m+1):
    a_perp.append(cochain({(V_basis[i-1],V_basis[i-2]):1,(V_basis[1],V_basis[0]):-1}))
    
    a_perp.append(cochain({(V_basis[i-1],V_basis[i-1]):1,(V_basis[0],V_basis[0]):i-2,
                          (V_basis[1],V_basis[1]):-Rational((i-2)*(2*m-1)+(2*m-2*i+1),2*m-3)}))
NVV_norm_cond=[ext_T_symb_elt({(N,):-1}).wedge(A) for A in a_perp]
norm_cond_basis+=NVV_norm_cond

In [ ]:
ortho_to_coboundary_image(NVV_norm_cond)

$\mathbb{R}\eta^*\wedge V^*\otimes \mathfrak{a}$-portion of normalization condition

In [ ]:
NVa_norm_cond=[cochain({(N,V_basis[i],gl2_basis[j]):-1}) 
               for i in range(len(V_basis)) for j in range(len(gl2_basis))]

norm_cond_basis+=NVa_norm_cond

In [ ]:
ortho_to_coboundary_image(NVa_norm_cond)

$\mathbb{R}\eta^*\wedge \mathbb{R}X^*\otimes V$-portion of normalization condition

In [ ]:
NXV_norm_cond=[cochain({(N,X,e1):-1})]
norm_cond_basis+=NXV_norm_cond

In [ ]:
ortho_to_coboundary_image(NXV_norm_cond)

$\mathbb{R}\eta^*\wedge \mathbb{R}X^*\otimes \mathfrak{a}$-portion of normalization condition

In [ ]:
NXa_norm_cond=[cochain({(N,X,V_basis[i]):-1}) for i in range(len(V_basis))]
norm_cond_basis+=NXa_norm_cond


In [ ]:
ortho_to_coboundary_image(NXa_norm_cond)

# Testing

In [ ]:
# Test antisymmetry

for A in T_symb_basis:
    for B in T_symb_basis:
        if ad(A,B)!=-ad(B,A):
            print('\nAsymmetry at ',(A,B),':')
            print('ad',(A,B),'=',ad(A,B))
            print('ad',(B,A),'=',ad(B,A))
print('antisymmetry test complete')

In [ ]:
# Test the Jacobi identity

for A in T_symb_basis:
    for B in T_symb_basis:
        for D in T_symb_basis:
            t1=ad(A,ad(B,D))
            t2=ad(ad(A,B),D)+ad(B,ad(A,D))
            if t1!=t2:
                print('Failure')
                print('ad(',A,'ad(',B,',',D,') ) =', ad(A,ad(B,D)))
                print('ad( ad(',A,',',B,') ) + ad(',B,', ad(',A,',',C,') )')
print('Jacobi test complete')

In [ ]:
# TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST 
#
# ad(se1,se2)
(0,0,0)
(0,N+3*X,0)
(X,Y,H)
(4*X+2*Y+e1,e2+e3+e6,10*e1+16*e2+4*e3+4*e4+10*e5-N)
(2*E-X+3*N,H-e3+5*N,2*X-2*e3+e4+20*N)
(e2,e5,N)
(e3,e4,-N)
(2*H+4*E+e4,2*e3-e4+N,4*e3-6*e4+10*N)


test_cases=[(0,0,0),(0,N+3*X,0),(X,Y,H),(4*X+2*Y+e1,e2+e3+e6,10*e1+16*e2+4*e3+4*e4+10*e5-N),
              (2*E-X+3*N,H-e3+5*N,2*X-2*e3+e4+20*N),(e2,e5,N),(e3,e4,-N),(2*H+4*E+e4,2*e3-e4+N,4*e3-6*e4+10*N)]

for trip in test_cases:
    # forward and backward tests
    ftest=(ad(trip[0],trip[1])==trip[2])
    rtest=(ad(trip[1],trip[0])==-trip[2])
    if not ftest:
        print('\nTriple', trip,'failed ftest:')
        print('ad({0},{1}) ='.format(trip[0],trip[1]), ad(trip[0],trip[1]))
        print('!=', trip[2])
    if not rtest:
        print('\nTriple', trip,'failed rtest:')
        print('ad({0},{1}) ='.format(trip[1],trip[0]), ad(trip[1],trip[0]))
        print('!=', -trip[2])

print('ad test complete')

In [ ]:
# TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST 
#
# ext_ad(se,ext)
# 
# To do: add more tests here for higher degrees


# deg 1
test_cases=[(0,0,0)]
test_cases.append((0,ext_T_symb_elt({(e2,):0}),0))
test_cases.append((0,ext_T_symb_elt({(E,):1,(H,):1,(e3,):4}),ext_T_symb_elt({})))
test_cases.append((X+e3,ext_T_symb_elt({}),0))
test_cases.append((X,ext_T_symb_elt({(e1,):1,(e2,):2}),ext_T_symb_elt({(e1,):-2})))
test_cases.append((Y,ext_T_symb_elt({(e2,):1}),ext_T_symb_elt({(e3,):-8})))
test_cases.append((H,ext_T_symb_elt({(e3,):1}),ext_T_symb_elt({(e3,):1})))
test_cases.append((E,ext_T_symb_elt({(N,):1}),ext_T_symb_elt({(N,):-2})))
test_cases.append((H,ext_T_symb_elt({(N,):1}),ext_T_symb_elt({})))
test_cases.append((X,ext_T_symb_elt({(N,):1}),0))
test_cases.append((Y,ext_T_symb_elt({(N,):1}),ext_T_symb_elt({})))
test_cases.append((e1+e2,ext_T_symb_elt({(N,):1}),ext_T_symb_elt({(e6,):1,(e5,):-1})))
test_cases.append((6*X-Y+e1,ext_T_symb_elt(({(e3,):1,(N,):1})),ext_T_symb_elt({(e2,):-6,(e4,):9,(e6,):1})))
test_cases.append((e2,ext_T_symb_elt({(e3,):1}),ext_T_symb_elt({(X,):1})))
test_cases.append((e2,ext_T_symb_elt({(e2,):1}),ext_T_symb_elt({(E,):1,(H,):-3})))
test_cases.append((e2,ext_T_symb_elt({(e1,):1}),ext_T_symb_elt({(Y,):5})))
test_cases.append((N,ext_T_symb_elt({(E,):1,(N,):-2,(e6,):1}),ext_T_symb_elt({(E,):-4})))
                  
for case in test_cases:
    ftest=(ext_ad(case[0],case[1])==case[2])
    if not ftest:
        print('\nCase',test_cases.index(case)+1, case,'failed ftest:')
        print('ad({0},{1}) ='.format(case[0],case[1]), ext_ad(case[0],case[1]))
        print('!=', case[2])
        
print('ext_ad test complete')

In [ ]:
type(list(coboundary_dict.keys())[0])

In [ ]:
def merge(dict1, dict2):
    result=copy.copy(dict2)
    result.update(dict1)
    return(result)

In [ ]:
# TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST 
#
# coboundary_dict
from sympy import Rational

c_test_dict={}
c_test_dict[(E,)]=cochain(merge({(ei,ei):-1 for ei in V_basis},{(N,N):-2}))
c_test_dict[(X,)]=cochain(merge({(V_basis[i],V_basis[i+1]):-1 for i in range(len(V_basis)-1)},{(H,X):2,(Y,H):-1}))
c_test_dict[(Y,)]=cochain(merge({(V_basis[i],V_basis[i-1]):-(i)*(2*m-i) for i in range(1,len(V_basis))},{(H,Y):-2,(X,H):1}))
c_test_dict[(e2,)]=cochain({(E,e2):1,(X,e3):1,(H,e2):(4-2*m-1),(Y,e1):(1)*(2*m-1),(V_basis[2*m-2-i],N):-1})


c_test_dict[(Y,Y)]=cochain(merge({(Y,Vbasis[i-1],Vbasis[i-2]):(i-1)*(2*m+1-i) for i in range(2,len(V_basis+1))}{(Y,H,Y):2,(Y,X,H):-1,(H,Y,Y):Rational(1,2)}))
c_test_dict[(Y,E)]=cochain(merge({(Y,Vbasis[i],Vbasis[i]):1 for i in range(len(V_basis))}{(Y,N,N):2,(H,Y,E):Rational(1,2)}))
c_test_dict[(Y,H)]=cochain(merge({(Y,Vbasis[i-1],Vbasis[i-1]):(2*i-2*m-1) for i in range(1,len(V_basis+1))}{(Y,X,X):2,(H,Y,H):Rational(1,2)}))
c_t

for key in c_test_dict:
    if c_test_dict[key]!=coboundary_dict[key]:
        print('\nFailure at', key,':')
        print('test value:', c_test_dict[key])
        print('actual value:', coboundary_dict[key])

In [ ]:
from sympy import Rational
c_test_dict[(Y,Y)]=cochain(merge({(Y,V_basis[i-1],V_basis[i-2]):(i-1)*(2*m+1-i) for i in range(2,len(V_basis)+1)},
                                 {(Y,H,Y):2,(Y,X,H):-1,(H,Y,Y):Rational(1,2)}))
c_test_dict[(Y,E)]=cochain(merge({(Y,V_basis[i],V_basis[i]):1 for i in range(len(V_basis))},
                                 {(Y,N,N):2,(H,Y,E):Rational(1,2)}))
c_test_dict[(Y,H)]=cochain(merge({(Y,V_basis[i-1],V_basis[i-1]):(2*i-2*m-1) for i in range(1,len(V_basis)+1)},
                                 {(Y,X,X):2,(H,Y,H):Rational(1,2)}))
c_test_dict[(H,X)]=cochain(merge({(H,V_basis[i],V_basis[i+1]):1 for i in range(len(V_basis)-1)},
                                 {(H,Y,H):1,(X,Y,X):-1}))
c_test_dict[(X,e1)]=cochain({(X,E,e1):-1,(X,H,e1):2*m-Rational(1,2),(X,V_basis[2*m-1],N):1,(E,V_basis[2*m-2],N):1})

c_test_dict[(E,e2)]=cochain({(E,X,e3):-1,(E,Y,e1):3-2*m,(E,H,e2):-2*m-1,(E,V_basis[2*m-2],N):-1})

c_test_dict[(H,e3)]=cochain({(H,X,e4):-1,(H,Y,e2):5-2*m,(H,E,e3):1,(H,V_basis[2*m-3],N):-1,(X,Y,e3):-1})

c_test_dict[(e2,Y)]=cochain(merge({(e2,X,H):-1,(e2,H,Y):2,(X,e1,Y):-1,(Y,e3,Y):-1,(H,e2,Y):Rational(1,2*m-3),(E,e2,Y):-1},
                                  {(e2,V_basis[i-1],V_basis[i-2]):(i-1)*(2*m+1-i) for i in range(3,2*m-1)}))
c_test_dict[(V_basis[2*m-1],Y)]=cochain(merge({(V_basis[2*m-1],Y,H):-1,(V_basis[2*m-1],H,X):2,(X,V_basis[2*m-2],X):-1,
                                               (H,V_basis[2*m-1],X):Rational(1,1-2*m),(E,V_basis[2*m-1],X):-1},
                                  {(V_basis[2*m-1],V_basis[i-1],V_basis[i]):1 for i in range(1,2*m-1)}))
